In [1]:
#!pip install rdflib networkx torch torch-geometric matplotlib

In [2]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import from_networkx

from rdflib.plugins.sparql import parser, algebra
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import pandas as pd
import json

/upb/users/g/ganeshk/profiles/unix/cs/.local/lib/python3.9/site-packages/transformers/utils/versions.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### 1) Example SPARQL Triples for Contrastive Learning

In [3]:
import csv
import json

def load_queries_from_csv(csv_path):
    dataset = []

    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            anchor = row["anchor_query"]

            # parse JSON lists stored as text
            positives = json.loads(row["positive_queries"])
            negatives = json.loads(row["negative_queries"])

            dataset.append({
                "id": row["id"],        # or int(row["id"]) if you want
                "anchor": anchor,
                "positives": positives,
                "negatives": negatives,
            })

    return dataset

# Example:
dataset = load_queries_from_csv("sample_sparql_queries-dataset.csv")
dataset = dataset[:1000]
print(dataset[0])


{'id': '0', 'anchor': 'PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\nPREFIX  owl:  <http://www.w3.org/2002/07/owl#>\nPREFIX  dbpr: <http://dbpedia.org/resource/>\n\nSELECT DISTINCT  *\nWHERE\n  { dbpr:Neural_networks\n              rdf:type  ?owl .\n    ?owl      rdf:type  owl:Class\n  }', 'positives': ['PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\nPREFIX  owl:  <http://www.w3.org/2002/07/owl#>\nPREFIX  dbpr: <http://dbpedia.org/resource/>\n\nSELECT DISTINCT  *\nWHERE\n  { dbpr:News_aggregator\n              rdf:type  ?owl .\n    ?owl      rdf:type  owl:Class\n  }', 'PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\nPREFIX  owl:  <http://www.w3.org/2002/07/owl#>\nPREFIX  dbpr: <http://dbpedia.org/resource/>\n\nSELECT DISTINCT  *\nWHERE\n  { dbpr:Computer_vision\n              rdf:type  ?owl .\n    ?owl      rdf:type  owl:Class\n  }', 'PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\nPREFIX  owl:  <http://www.w3.org/2002/07/owl#>\nPR

 ------------------------------------------------------------
### 2) Convert SPARQL Algebra → NetworkX Graph
 ------------------------------------------------------------


In [4]:
def algebra_to_graph(op, G=None, parent=None, node_id=0):
    """
    Recursively builds a directed graph from the SPARQL algebra tree.
    Each algebra operator becomes a node, and BGP triples become leaf nodes.
    """
    if G is None:
        G = nx.DiGraph()

    # Label of the current algebra operator
    node_label = op.name if hasattr(op, 'name') else type(op).__name__
    node_name = f"{node_label}_{node_id}"

    G.add_node(node_name, label=node_label)

    if parent is not None:
        G.add_edge(parent, node_name)

    # Special case: Basic Graph Pattern (BGP) triples
    if getattr(op, "name", None) == "BGP":
        for s, p, o in op.triples:
            triple_node = f"Triple_{s}_{p}_{o}_{node_id}"
            triple_label = f"{s} {p} {o}"
            G.add_node(triple_node, label=triple_label)
            G.add_edge(node_name, triple_node)

    # Recurse over attributes/values in the algebra object
    # to find nested algebra operations
    for v in op.values():
        if hasattr(v, "name"):  # nested algebra op
            node_id += 1
            G = algebra_to_graph(v, G, node_name, node_id)
        elif isinstance(v, list):
            for sub in v:
                if hasattr(sub, "name"):
                    node_id += 1
                    G = algebra_to_graph(sub, G, node_name, node_id)

    return G


### 3) Convert NetworkX → PyTorch Geometric Data

In [5]:
def graph_to_pyg(G):
    labels = nx.get_node_attributes(G, "label")

    # Simple label hashing → 3D feature vector per node
    feats = []
    for lbl in labels.values():
        h = abs(hash(lbl)) % (10**6)
        vec = np.array([
            h % 97,
            (h // 97) % 97,
            (h // (97**2)) % 97
        ], dtype=float)
        feats.append(vec / 100.0)

    feats = np.stack(feats, axis=0)
    feats = torch.tensor(feats, dtype=torch.float)

    data = from_networkx(G)
    # Ensure we have node features in correct order
    data.x = feats
    # Single-graph batch index (all zeros)
    data.batch = torch.zeros(data.num_nodes, dtype=torch.long)

    return data


### 4) Wrapper: SPARQL Query → PyG Graph

In [6]:
def encode_sparql_query_to_graph(query_str):
    # Parse and translate to algebra
    parsed = parser.parseQuery(query_str)
    alg = algebra.translateQuery(parsed).algebra

    # Algebra → NetworkX
    G = algebra_to_graph(alg)

    # NetworkX → PyG
    pyg_data = graph_to_pyg(G)
    return pyg_data, G



### 6) GCN Encoder

In [7]:
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_dim=3, hidden=32, out_dim=16):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, out_dim)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        # aggregate node embeddings to graph embedding
        x = global_mean_pool(x, batch)
        return x  # shape: [batch_size, out_dim]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GCNEncoder().to(device)
print(model)

# ------------------------------------------------------------
# 7) Helper: Query → Embedding (with current model)
# ------------------------------------------------------------

def get_embedding(model, query_str):
    data, _ = encode_sparql_query_to_graph(query_str)
    data = data.to(device)
    model.eval()
    with torch.no_grad():
        emb = model(data.x, data.edge_index, data.batch)
    return emb  # shape [1, out_dim]


GCNEncoder(
  (conv1): GCNConv(3, 32)
  (conv2): GCNConv(32, 16)
)


### 8) InfoNCE Contrastive Loss

In [8]:
def info_nce(anchor, positives, negatives, temperature=0.1):
    """
    anchor:     [1, d]
    positives:  list of [1, d]
    negatives:  list of [1, d]
    """
    # Normalize
    anchor = anchor / anchor.norm(dim=-1, keepdim=True)
    pos = torch.cat(positives, dim=0)  # [P, d]
    neg = torch.cat(negatives, dim=0)  # [N, d]

    pos = pos / pos.norm(dim=-1, keepdim=True)
    neg = neg / neg.norm(dim=-1, keepdim=True)

    # Similarity to anchor (dot product)
    pos_sim = (pos @ anchor.T) / temperature   # [P, 1]
    neg_sim = (neg @ anchor.T) / temperature   # [N, 1]

    # Combine logits: first P are positives, then N negatives
    logits = torch.cat([pos_sim, neg_sim], dim=0).T  # [1, P+N]
    # Label index of the correct class = 0..P-1 (we treat "any positive" as target 0; simplest demo)
    # For a simple demo, we mark one representative positive as the target.
    target = torch.tensor([0], dtype=torch.long, device=logits.device)

    loss = F.cross_entropy(logits, target)
    return loss, logits.detach().cpu()


### 10) Mini Contrastive Training Loop

In [9]:
graph_cache = {}

def get_graph_cached(query):
    if query not in graph_cache:
        pyg, _ = encode_sparql_query_to_graph(query)
        graph_cache[query] = pyg
    return graph_cache[query]

In [10]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs=20

for epoch in range(1, num_epochs + 1):
    total_loss = 0

    for item in dataset:
        anchor_query = item["anchor"]
        positive_queries = item["positives"]
        negative_queries = item["negatives"]

        # Encode data
        data_anchor, _ = encode_sparql_query_to_graph(anchor_query)
        data_anchor = data_anchor.to(device)

        pos_data = [get_graph_cached(q).to(device)
                    for q in positive_queries]
        neg_data = [get_graph_cached(q).to(device)
                    for q in negative_queries]

        model.train()
        optimizer.zero_grad()

        anchor_emb = model(data_anchor.x, data_anchor.edge_index, data_anchor.batch)
        pos_embs = [model(d.x, d.edge_index, d.batch) for d in pos_data]
        neg_embs = [model(d.x, d.edge_index, d.batch) for d in neg_data]

        loss, _ = info_nce(anchor_emb, pos_embs, neg_embs)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} | Avg Loss: {total_loss / len(dataset):.4f}")



Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x7fcb09122310>
Traceback (most recent call last):
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/rdflib/xsd_datetime.py", line 587, in parse_xsd_date
    return parse_date(date_string if not minus else ("-" + date_string))
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/isodate/isodates.py", line 193, in parse_date
    raise ISO8601Error("Unrecognised ISO 8601 date format: %r" % datestring)
isodate.isoerror.ISO8601Error: Unrecognised ISO 8601 date format: rdflib.term.Literal('101-01-01')
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/X

Epoch 1 | Avg Loss: 1.3358


Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x7fcb09122310>
Traceback (most recent call last):
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/rdflib/xsd_datetime.py", line 587, in parse_xsd_date
    return parse_date(date_string if not minus else ("-" + date_string))
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/isodate/isodates.py", line 193, in parse_date
    raise ISO8601Error("Unrecognised ISO 8601 date format: %r" % datestring)
isodate.isoerror.ISO8601Error: Unrecognised ISO 8601 date format: '-1700-01-01'


Epoch 2 | Avg Loss: 1.2794


Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x7fcb09122310>
Traceback (most recent call last):
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/rdflib/xsd_datetime.py", line 587, in parse_xsd_date
    return parse_date(date_string if not minus else ("-" + date_string))
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/isodate/isodates.py", line 193, in parse_date
    raise ISO8601Error("Unrecognised ISO 8601 date format: %r" % datestring)
isodate.isoerror.ISO8601Error: Unrecognised ISO 8601 date format: '-1700-01-01'


...

Epoch 18 | Avg Loss: 1.2254


Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x7fcb09122310>
Traceback (most recent call last):
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/rdflib/xsd_datetime.py", line 587, in parse_xsd_date
    return parse_date(date_string if not minus else ("-" + date_string))
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/isodate/isodates.py", line 193, in parse_date
    raise ISO8601Error("Unrecognised ISO 8601 date format: %r" % datestring)
isodate.isoerror.ISO8601Error: Unrecognised ISO 8601 date format: '-1700-01-01'


Epoch 19 | Avg Loss: 1.2247


Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x7fcb09122310>
Traceback (most recent call last):
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/rdflib/xsd_datetime.py", line 587, in parse_xsd_date
    return parse_date(date_string if not minus else ("-" + date_string))
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/isodate/isodates.py", line 193, in parse_date
    raise ISO8601Error("Unrecognised ISO 8601 date format: %r" % datestring)
isodate.isoerror.ISO8601Error: Unrecognised ISO 8601 date format: '-1700-01-01'


Epoch 20 | Avg Loss: 1.2291


### 11) Inspect Similarities After Training

In [11]:
from statistics import mean

model.eval()

def cosine(a, b):
    a = a / a.norm(dim=-1, keepdim=True)
    b = b / b.norm(dim=-1, keepdim=True)
    return float(a @ b.T)

pos_sims = []
neg_sims = []

for idx, example in enumerate(dataset):
    anchor_query     = example["anchor"]
    positive_queries = example["positives"]
    negative_queries = example["negatives"]

    # Skip empty cases
    if not positive_queries or not negative_queries:
        continue

    # Compute embeddings
    anchor_emb = get_embedding(model, anchor_query)
    pos_embs   = [get_embedding(model, q) for q in positive_queries]
    neg_embs   = [get_embedding(model, q) for q in negative_queries]

    # Stats collection
    best_pos = max(cosine(anchor_emb, e) for e in pos_embs)
    best_neg = max(cosine(anchor_emb, e) for e in neg_embs)

    pos_sims.append(best_pos)
    neg_sims.append(best_neg)

    if idx < 10:
        print(f"\n--- Example {idx} ---")
        print("Anchor Query:")
        print(anchor_query.strip())

        print("\nPositive similarities:")
        for i, e in enumerate(pos_embs):
            print(f"  Positive {i}: {cosine(anchor_emb, e):.4f}")

        print("\nNegative similarities:")
        for j, e in enumerate(neg_embs):
            print(f"  Negative {j}: {cosine(anchor_emb, e):.4f}")

print("\n=== Aggregate Stats for All Samples ===")
print(f"Mean positive sim: {mean(pos_sims):.3f}")
print(f"Mean negative sim: {mean(neg_sims):.3f}")
print(f"Fraction where pos > neg: "
      f"{sum(p > n for p, n in zip(pos_sims, neg_sims)) / len(pos_sims):.3f}")



--- Example 0 ---
Anchor Query:
PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  owl:  <http://www.w3.org/2002/07/owl#>
PREFIX  dbpr: <http://dbpedia.org/resource/>

SELECT DISTINCT  *
WHERE
  { dbpr:Neural_networks
              rdf:type  ?owl .
    ?owl      rdf:type  owl:Class
  }

Positive similarities:
  Positive 0: 0.9952
  Positive 1: 0.9967
  Positive 2: 0.9991
  Positive 3: 0.9963
  Positive 4: 0.9968

Negative similarities:
  Negative 0: -0.1741
  Negative 1: 0.8259
  Negative 2: 0.7810
  Negative 3: -0.1501
  Negative 4: -0.1928

--- Example 1 ---
Anchor Query:
PREFIX  rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX  owl:  <http://www.w3.org/2002/07/owl#>
PREFIX  dbpr: <http://dbpedia.org/resource/>

SELECT DISTINCT  *
WHERE
  { dbpr:Plateau  rdf:type  ?owl .
    ?owl      rdf:type  owl:Class
  }

Positive similarities:
  Positive 0: 0.9994

Negative similarities:
  Negative 0: 0.2082
  Negative 1: 0.0237
  Negative 2: 0.0237
  Negative 3: 0.52

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x7fcb09122310>
Traceback (most recent call last):
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/rdflib/xsd_datetime.py", line 587, in parse_xsd_date
    return parse_date(date_string if not minus else ("-" + date_string))
  File "/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/myenv/lib/python3.9/site-packages/isodate/isodates.py", line 193, in parse_date
    raise ISO8601Error("Unrecognised ISO 8601 date format: %r" % datestring)
isodate.isoerror.ISO8601Error: Unrecognised ISO 8601 date format: rdflib.term.Literal('101-01-01')
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/X


=== Aggregate Stats for All Samples ===
Mean positive sim: 0.997
Mean negative sim: 0.659
Fraction where pos > neg: 0.987
